# W10 — Inference-to-Prediction Workbench

Advanced Machine Learning · November 3, 2026 · independent CPU notebook.
Save a copy in Drive and run from the top. Tiny teaching cases are created automatically; no upload, key, GPU or paid service is required.

[Main note](https://github.com/lunalab-ai/advML/blob/2026-fall-w10/course/notion/w10/w10-inference-to-prediction.md) · [Original-question workbook](https://github.com/lunalab-ai/advML/blob/2026-fall-w10/course/handouts/w10-original-question-workbook.pdf) · [Code guide](https://github.com/lunalab-ai/advML/blob/2026-fall-w10/course/notion/w10/w10-code-guide.md)

Predict before running. Core tasks E1–E6 have response fields separate from demonstrations; an unfinished answer never substitutes a hidden completed answer. Original-exam calculations belong in the separate blank workbook.


In [ ]:
import sys, subprocess, importlib.metadata, hashlib, urllib.request, time
from pathlib import Path

PINS = {"numpy":"2.2.6","matplotlib":"3.10.3","scipy":"1.16.3","pandas":"2.3.3","gradio":"6.27.0"}
missing=[]
for package,version in PINS.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version: missing.append(f"{package}=={version}")
restart=[p for p,v in PINS.items() if p in sys.modules and getattr(sys.modules[p],"__version__",v)!=v]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])
if restart:
    raise RuntimeError("Packages updated. Use Runtime > Restart session, then Run all. Loaded old packages: "+", ".join(restart))
CODE_REF="2026-fall-w10"
EXPECTED={'advml_review.py': 'ea9c7fe9a6ce96c513496bc08271a6af2d739a6009223e9ff7a66e66e9b04327', 'advml_review_app.py': '745271b5db60e28761415f764fd594dce1e7cc57313ef2f3bc5356a987bbe89f', 'advml_sampling.py': '7b881dd17a5895eb6c44835ae6e6516e2448258f99a76965a6b9d352049b0f1e', 'advml_vi.py': '699c3e5f7f75bee9dfdceff05e4956f7bd2bf0b9abff6c0eacf11a9ac3f1ad2a', 'advml_predictive.py': 'beaa85083c3f1cc4ead5f253ff11c82b7f71dc3ad2da8fc21d7307505e20583b'}
local=Path.cwd()/"src"
local_ok=all((local/name).is_file() and hashlib.sha256((local/name).read_bytes()).hexdigest()==digest for name,digest in EXPECTED.items())
support=local if local_ok else Path.cwd()/"w10_support"/CODE_REF
support.mkdir(parents=True,exist_ok=True)
downloaded=[]
for name,digest in EXPECTED.items():
    path=support/name
    if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==digest:continue
    url=f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/{name}"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url,timeout=40) as response:payload=response.read()
            break
        except (OSError,TimeoutError):
            if attempt==2:raise
            time.sleep(attempt+1)
    if hashlib.sha256(payload).hexdigest()!=digest:
        raise RuntimeError("Source hash mismatch; do not run unverified code: "+name)
    temporary=path.with_suffix(".download");temporary.write_bytes(payload);temporary.replace(path);downloaded.append(name)
sys.path.insert(0,str(support.resolve()))
import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display,Markdown
from advml_review import (discrete_importance,gaussian_loss,paired_runs,scalar_posterior,
                          scalar_prediction,shared_weight_covariance)
from advml_review_app import build_app,loss_view,bayes_view
IN_COLAB="google.colab" in sys.modules
print({"Python":sys.version.split()[0],"code_ref":CODE_REF,"downloaded":downloaded,
       "cache_or_reviewed_local":not downloaded,"versions":{p:importlib.metadata.version(p) for p in PINS},
       "data":"transparent synthetic teaching cases"})


## Inspect the actual reusable definitions

The algorithms remain in public generic helpers. This notebook supplies experiments, predictions, interpretation and short tasks. The code guide states inputs, return values, side effects and variance conventions.


In [ ]:
import inspect
functions=[discrete_importance,gaussian_loss,paired_runs,scalar_posterior,scalar_prediction,shared_weight_covariance,build_app]
display(Markdown("\n".join(f"- [{f.__name__}](https://github.com/lunalab-ai/advML/blob/{CODE_REF}/src/{Path(inspect.getsourcefile(f)).name}#L{inspect.getsourcelines(f)[1]})" for f in functions)))


## 1. Target, proposal and two denominators

Reuse the W5 teaching case. The state values are 0,1,2; both probability vectors sum to one. Every draw gets a row, including repeated values. Direct IS uses the draw count, while SNIS uses raw-weight mass.


In [ ]:
worked_is=discrete_importance([.2,.3,.5],[.5,.25,.25],[0,1,1,2])
display(worked_is["table"])
print({k:worked_is[k] for k in ("exact","numerator","weight_sum","direct_is","snis","weight_ess")})


### E1 — Predict before computing (core)

Use target [0.4,0.6], proposal [0.25,0.75] and fixed draws [1,0,1].
Predict the three raw weights in draw order. Explain the target/proposal roles and why both occurrences of state1 remain.
Hint: identify the state first, then look up both masses. Self-check: your answer must have three entries.


In [ ]:
e1_prediction=None
e1_explanation=""
print("E1 response:",e1_prediction,e1_explanation)


### E2 — Fill the denominators and explain (core)

Three iid proposal draws have raw p/q weights [.5,1.5,2] and function values [2,1,3]. Assume the target is normalized.
The shared weighted numerator is 8.5. Fill each denominator and explain why a normalized-weight sum receives no extra division by the draw count.
Hint: distinguish count from mass. Self-check: reconstruct each estimator's definition before calculating.


In [ ]:
e2_direct_denominator=None
e2_snis_denominator=None
e2_reason=""
print("E2 response:",e2_direct_denominator,e2_snis_denominator,e2_reason)


## 2. Summed Gaussian NLL and a shape error

Reuse W7 observations [1,2], means [1.5,1.5] and variance1. Retain the normalization term and sum over observations. A noise SD must be squared before passing it as variance.


In [ ]:
worked_loss=gaussian_loss([1,2],[1.5,1.5],1)
display(worked_loss["table"])
print({k:worked_loss[k] for k in ("sse","mse","sum_nll")})
y_debug=np.array([2.,4.]);prediction_column=np.array([[1.],[3.]])
print("Debug shapes:",y_debug.shape,prediction_column.shape,
      "naive subtraction shape:",(y_debug-prediction_column).shape)
try:
    gaussian_loss(y_debug,prediction_column,.5)
except ValueError as exc:
    print("Expected shape rejection:",exc)
else:
    raise AssertionError("The strict shape check did not reject broadcasting.")


### E3 — Repair the array shape (core)

Use y_debug and prediction_column above. Produce one prediction per observation, then calculate SSE, MSE and summed NLL with variance0.5.
Explain why the original subtraction made four comparisons.
Hint: inspect dimensions before choosing a transformation. Self-check: the repaired array must match y_debug.shape exactly.


In [ ]:
e3_mean=None
e3_result=None
e3_explanation=""
print("E3 response:",e3_mean,e3_result,e3_explanation)


## 3. Separate experiment records: use keys

These records are a separate synthetic experiment, not the preceding residual table. Match the same run IDs before subtracting A-B. A positive difference favors B; an exact tie is not a win.


In [ ]:
worked_a=pd.DataFrame({"run_id":["r1","r2","r3"],"error":[.2,.6,.4]})
worked_b=pd.DataFrame({"run_id":["r3","r1","r2"],"error":[.2,.6,.4]})
worked_pairs=paired_runs(worked_a,worked_b)
display(worked_pairs["table"])
print({k:worked_pairs[k] for k in ("mean_difference","a_wins","b_wins","ties")})
print("Wrong row-position differences:",worked_a.error.to_numpy()-worked_b.error.to_numpy())


### E4 — Predict the full ID-labelled vector (core)

A records are(s1,.8),(s2,.2),(s3,.4). B rows are(s2,.3),(s3,.1),(s1,.6).
Predict each matched A-B difference in A's ID order, then the mean and B-win count.
Explain why matching a mean or a win count alone does not prove ID alignment.
Hint: make a two-column keyed table. Self-check: permuting B's rows must preserve every ID-specific difference.


In [ ]:
e4_differences=None
e4_mean=None
e4_b_wins=None
print("E4 response:",e4_differences,e4_mean,e4_b_wins)


## 4. Weight posterior, latent prediction and a new response

Reuse W7: prior variance1, known noise SD1, data(1,1),(2,2), no intercept. Complete the square or add precisions. The prediction at x*=3 has two distinct variances.


In [ ]:
worked_posterior=scalar_posterior([1,2],[1,2],prior_variance=1,noise_sd=1)
worked_prediction=scalar_prediction(worked_posterior,3)
print("Posterior and quadratic coefficients:",worked_posterior)
print("Two prediction queries:",worked_prediction)
prior_covariance=shared_weight_covariance([-2,1],1)
posterior_covariance=shared_weight_covariance([-2,1],worked_posterior["variance"])
display(pd.DataFrame(prior_covariance,index=["x=-2","x=1"],columns=["x=-2","x=1"]))
print("Prior covariance eigenvalues:",np.linalg.eigvalsh(prior_covariance))
print("Conditioned latent covariance:",posterior_covariance)


### E5 — Keep the two variances separate (core)

For this transfer task, a posterior is supplied: w|D~N(0.4,0.3). Let x*=-2 and new independent noise SD0.5.
Give the latent mean, latent variance and new-response variance. Explain the squared input and the one additional variance.
Hint: the second Normal argument is variance. Self-check: the variance difference equals the squared noise SD.


In [ ]:
e5_mean=None
e5_latent_variance=None
e5_response_variance=None
print("E5 response:",e5_mean,e5_latent_variance,e5_response_variance)


### E6 — Connect your own diagnostic (core)

Implement my_diagnostic(y,mean) to return RMSE for matching finite one-dimensional arrays. Reject a shape mismatch. Connect it through build_app(diagnostic=my_diagnostic), then test two different mean inputs.
The function currently returns None, and the app explicitly labels it unfinished. No hidden completed diagnostic is used.
Hint: define the residual vector, choose the appropriate reduction, and inspect the callback argument order. Self-check: compare your output against a hand calculation.
Optional extension: compare a different diagnostic and explain what it measures.


In [ ]:
def my_diagnostic(y,mean):
    # Replace this body with your implementation, then rebuild the app cell.
    return None


## 5. Build, inspect and use the three-tab app

The app reuses the functions inspected above. Compare weights and denominators; inspect residuals and separate keyed run records; compare exact Bayesian prediction with a separately labeled Monte Carlo estimate.

After editing E6, run this cell again to connect the updated callable. In Colab, open the printed APP_URL in a new tab. If a formatted output has a display error, the plain URL is still the entry point. Keep the notebook link as the permanent resource; a shared app URL is temporary.


In [ ]:
demo=build_app(diagnostic=my_diagnostic)
if IN_COLAB:
    demo.launch(share=True,inline=False)
    print("APP_URL",demo.share_url)
else:
    print("Local validation constructed the app and callbacks; actual browser interaction is checked separately.")
probe=loss_view("1,2","1.5,1.5","1,2",1,my_diagnostic)[0]
print("Current custom diagnostic:",probe["custom_diagnostic_A"])


## 6. Explain what changed

Change only Monte Carlo draws while keeping data/scales fixed. The exact posterior and exact predictive variances should stay constant. Then change noise SD and inspect the new-response variance difference.
Export a CSV and check that its values agree with the displayed current results.
Finish the six core responses and all nine original-question workbook subparts. These are formative class activities, not a new graded assignment.


In [ ]:
assert np.isclose(worked_is["direct_is"],1.6) and np.isclose(worked_is["snis"],4/3)
assert np.isclose(worked_is["weight_ess"],36/11)
assert worked_loss["sse"]==.5 and np.isclose(worked_loss["sum_nll"],-norm.logpdf([1,2],loc=[1.5,1.5],scale=1).sum())
np.testing.assert_allclose(worked_pairs["table"].difference_A_minus_B,[-.4,.2,.2])
assert worked_pairs["b_wins"]==2 and worked_pairs["ties"]==0
np.testing.assert_allclose([worked_posterior["mean"],worked_posterior["variance"]],[5/6,1/6])
np.testing.assert_allclose([worked_prediction["mean_f"],worked_prediction["variance_f"],worked_prediction["variance_new_y"]],[2.5,1.5,2.5])
np.testing.assert_allclose(np.linalg.eigvalsh(prior_covariance),[0,5],atol=1e-12)
mc10=bayes_view("1,2","1,2",1,1,3,10,7)[0]
mc1000=bayes_view("1,2","1,2",1,1,3,1000,7)[0]
assert mc10["exact_prediction"]==mc1000["exact_prediction"]
assert np.isclose(mc10["MC_standard_error_of_mean_f"]/mc1000["MC_standard_error_of_mean_f"],10)
print("W10_INFERENCE_PREDICTION_RUNTIME_PASS","synthetic","exercise fields are not completion claims")
